# Public reproducibility copy

This notebook is copied from the authoritative thesis evidence. Machine-specific paths were replaced by documented placeholders; scientific code, parameters, and reported values were not changed. External datasets, lexicons, and model files are not bundled.


# Notebook 4 – Arabic Query Preprocessing and Automatic Query Term Selection (DD-001)

## Objective

This notebook implements the first stage of the proposed retrieval framework by preprocessing the QuranQA user questions and selecting the most informative query terms.

The preprocessing pipeline prepares the questions for lexical semantic query expansion by:

1. Loading the QuranQA dataset.
2. Normalizing Arabic text.
3. Removing punctuation and unnecessary symbols.
4. Tokenizing the questions.
5. Filtering non-content words.
6. Selecting candidate query terms for lexical semantic expansion.

The output of this notebook will serve as the input to **Notebook 5**, where the selected terms will be mapped to their Qur'anic roots using the Quranic Arabic Corpus (QAC).

In [1]:
# ============================================================
# Configure Project Path
# ============================================================

import os
import sys
from pathlib import Path


def find_workspace_root(start=None):
    """
    Locate the current thesis workspace without hard-coding a user-specific
    absolute Windows path.

    An optional THESIS_WORKSPACE_ROOT environment variable can be used when
    the notebook server is started outside the thesis workspace.
    """

    configured = os.environ.get("THESIS_WORKSPACE_ROOT")

    if configured:
        configured_path = Path(configured).expanduser().resolve()

        if (configured_path / "02_Methodology_Evidence").is_dir():
            return configured_path

        raise FileNotFoundError(
            "THESIS_WORKSPACE_ROOT is set, but it does not contain "
            "'02_Methodology_Evidence'."
        )

    start = Path(start or Path.cwd()).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "02_Methodology_Evidence").is_dir():
            return candidate

        expected_child = (
            candidate /
            "Master Thesis - Quranic Passage Retrieval - Writing Workspace"
        )

        if (expected_child / "02_Methodology_Evidence").is_dir():
            return expected_child

    raise FileNotFoundError(
        "Could not locate the thesis workspace. Start Jupyter from the "
        "workspace (or one of its subfolders), or set THESIS_WORKSPACE_ROOT."
    )


PROJECT_ROOT = find_workspace_root()

METHODOLOGY_DIR = PROJECT_ROOT / "02_Methodology_Evidence"

QAC_ROOT_EXTRACTION_DIR = (
    METHODOLOGY_DIR /
    "QAC and Root Extraction"
)

# The reusable preprocessing module is stored beside this notebook
if str(QAC_ROOT_EXTRACTION_DIR) not in sys.path:
    sys.path.insert(0, str(QAC_ROOT_EXTRACTION_DIR))

print("Project Root:")
print(PROJECT_ROOT)

print("\nQAC / Root Extraction Directory:")
print(QAC_ROOT_EXTRACTION_DIR)


Project Root:
${THESIS_WORKSPACE}

QAC / Root Extraction Directory:
${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction


In [2]:
# ============================================================
# Standard Library
# ============================================================

from pathlib import Path
from collections import Counter
import re

# ============================================================
# Third-Party Libraries
# ============================================================

import pandas as pd

In [3]:
# ============================================================
# Import Arabic Preprocessing Module
# ============================================================

import arabic_preprocessing as anlp

print("✅ Arabic preprocessing module imported successfully.")


✅ Arabic preprocessing module imported successfully.


In [4]:
# ============================================================
# Configure Display Settings
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 120)

print("✅ Display settings configured successfully.")

✅ Display settings configured successfully.


In [5]:
# ============================================================
# Define Project Paths
# ============================================================

# QuranQA benchmark resources
QURANQA_DIR = (
    METHODOLOGY_DIR /
    "Data and Resources" /
    "QuranQA"
)

# Input files
TRAIN_FILE = QURANQA_DIR / "QQA23_TaskA_ayatec_v1.2_train.tsv"
DEV_FILE   = QURANQA_DIR / "QQA23_TaskA_ayatec_v1.2_dev.tsv"
TEST_FILE  = QURANQA_DIR / "QQA23_TaskA_ayatec_v1.2_test.tsv"

# Point-2 corrected outputs are kept separate from preserved historical outputs
OUTPUT_DIR = (
    QAC_ROOT_EXTRACTION_DIR /
    "Corrected_Outputs" /
    "Point_2_Query_Root_Provenance"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Display paths
print("=" * 70)
print("PROJECT PATHS")
print("=" * 70)
print(f"Project Root       : {PROJECT_ROOT}")
print(f"QuranQA Directory  : {QURANQA_DIR}")
print(f"Output Directory   : {OUTPUT_DIR}")

print("\nInput Validation")
print(f"Train exists       : {TRAIN_FILE.exists()}")
print(f"Development exists : {DEV_FILE.exists()}")
print(f"Test exists        : {TEST_FILE.exists()}")
print("=" * 70)


PROJECT PATHS
Project Root       : ${THESIS_WORKSPACE}
QuranQA Directory  : ${THESIS_WORKSPACE}\02_Methodology_Evidence\Data and Resources\QuranQA
Output Directory   : ${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\Point_2_Query_Root_Provenance

Input Validation
Train exists       : True
Development exists : True
Test exists        : True


In [6]:
# ============================================================
# Load the QuranQA Datasets
# ============================================================

# Load each dataset (files do not contain column headers)
train_df = pd.read_csv(
    TRAIN_FILE,
    sep="\t",
    header=None,
    names=["Question_ID", "Question"]
)

dev_df = pd.read_csv(
    DEV_FILE,
    sep="\t",
    header=None,
    names=["Question_ID", "Question"]
)

test_df = pd.read_csv(
    TEST_FILE,
    sep="\t",
    header=None,
    names=["Question_ID", "Question"]
)

# Display dataset sizes
dataset_summary = pd.DataFrame({
    "Dataset": ["Training", "Development", "Test"],
    "Questions": [
        len(train_df),
        len(dev_df),
        len(test_df)
    ]
})

print("=" * 60)
print("QURANQA DATASETS LOADED")
print("=" * 60)

display(dataset_summary)

print(f"\nTotal Questions: {dataset_summary['Questions'].sum()}")

QURANQA DATASETS LOADED


,Dataset,Questions
0,Training,174
1,Development,25
2,Test,52



Total Questions: 251


In [7]:
# ============================================================
# Combine the QuranQA Datasets
# ============================================================

# Add dataset labels
train_df["Dataset"] = "Train"
dev_df["Dataset"] = "Development"
test_df["Dataset"] = "Test"

# Combine all questions
questions_df = pd.concat(
    [train_df, dev_df, test_df],
    ignore_index=True
)

# Reorder columns
questions_df = questions_df[
    ["Dataset", "Question_ID", "Question"]
]

print("=" * 60)
print("COMBINED QUESTION DATASET")
print("=" * 60)

print(f"Total Questions : {len(questions_df)}")

display(questions_df.head(10))

COMBINED QUESTION DATASET
Total Questions : 251


,Dataset,Question_ID,Question
0,Train,101,من هم قوم شعيب؟
1,Train,102,من هم قوم موسى؟
2,Train,103,من بنى الكعبة؟
3,Train,105,من هو النبي المعروف بالصبر؟
4,Train,106,من كفل السيدة مريم؟
5,Train,107,ما معنى الحطمة؟
6,Train,108,من هو اخو سيدنا موسى؟
7,Train,109,ما معنى القارعة؟
8,Train,110,ما معنى الجاثية؟
9,Train,111,من هم الاسباط؟


## DD-001 – Arabic Text Normalization

The first preprocessing stage consists of Arabic text normalization.

Arabic text may contain orthographic variations that refer to the same lexical item. These variations can reduce lexical matching during information retrieval if they are not normalized consistently.

The normalization process implemented in this study applies the following transformations:

- Normalize different forms of Alif (أ، إ، آ، ٱ → ا)
- Normalize Alif Maqsura (ى → ي)
- Remove Tatweel (ـ)
- Remove Arabic diacritics (Harakat)

Unlike many general-purpose Arabic NLP pipelines, this study intentionally preserves **Ta Marbuta (ة)**. Since the subsequent stages rely on lexical matching with the **Quranic Arabic Corpus (QAC)** and **Maqāyīs al-Lughah**, preserving the canonical spelling reduces the risk of altering lexical forms before root extraction.

The objective is to reduce orthographic variation while preserving the lexical forms required for accurate root extraction and lexical semantic query expansion.

## Applying Arabic Text Normalization

After defining the normalization procedure, the next step is to apply it to every question in the QuranQA dataset.

The normalized questions are stored in a new column while preserving the original questions. Keeping both versions enables comparison, validation, and reproducibility throughout the remaining stages of the retrieval pipeline.

In [8]:
# ============================================================
# Apply DD-001: Arabic Text Normalization
# ============================================================

# Create a copy to preserve the original dataset
normalized_df = questions_df.copy()

# Apply normalization
normalized_df["Normalized_Question"] = (
    normalized_df["Question"]
    .apply(anlp.normalize_arabic)
)

print("=" * 70)
print("DD-001 COMPLETED: ARABIC TEXT NORMALIZATION")
print("=" * 70)

print(f"Total Questions Processed : {len(normalized_df)}")

display(
    normalized_df[
        [
            "Question_ID",
            "Question",
            "Normalized_Question"
        ]
    ].head(10)
)

DD-001 COMPLETED: ARABIC TEXT NORMALIZATION
Total Questions Processed : 251


,Question_ID,Question,Normalized_Question
0,101,من هم قوم شعيب؟,من هم قوم شعيب؟
1,102,من هم قوم موسى؟,من هم قوم موسي؟
2,103,من بنى الكعبة؟,من بني الكعبة؟
3,105,من هو النبي المعروف بالصبر؟,من هو النبي المعروف بالصبر؟
4,106,من كفل السيدة مريم؟,من كفل السيدة مريم؟
5,107,ما معنى الحطمة؟,ما معني الحطمة؟
6,108,من هو اخو سيدنا موسى؟,من هو اخو سيدنا موسي؟
7,109,ما معنى القارعة؟,ما معني القارعة؟
8,110,ما معنى الجاثية؟,ما معني الجاثية؟
9,111,من هم الاسباط؟,من هم الاسباط؟


In [9]:
# ============================================================
# Evaluate the Effect of Arabic Text Normalization
# ============================================================

# Identify questions modified by normalization
changed_questions = normalized_df[
    normalized_df["Question"] != normalized_df["Normalized_Question"]
].copy()

# Calculate statistics
total_questions = len(normalized_df)
changed_count = len(changed_questions)
unchanged_count = total_questions - changed_count
changed_percentage = round((changed_count / total_questions) * 100, 2)

# ------------------------------------------------------------
# Create summary table
# ------------------------------------------------------------

summary_df = pd.DataFrame({
    "Metric": [
        "Total Questions",
        "Changed Questions",
        "Unchanged Questions",
        "Percentage Changed"
    ],
    "Value": [
        total_questions,
        changed_count,
        unchanged_count,
        f"{changed_percentage}%"
    ]
})

print("=" * 70)
print("DD-001 VALIDATION: ARABIC TEXT NORMALIZATION")
print("=" * 70)

display(summary_df)

# ------------------------------------------------------------
# Display representative examples
# ------------------------------------------------------------

print("\nRepresentative Examples of Normalized Questions\n")

display(
    changed_questions[
        ["Question_ID", "Question", "Normalized_Question"]
    ].head(10)
)

# ------------------------------------------------------------
# Interpretation
# ------------------------------------------------------------

print("\nInterpretation:")

if changed_count == 0:
    print("- No questions required normalization.")
else:
    print(
        f"- Arabic text normalization modified "
        f"{changed_count} out of {total_questions} questions "
        f"({changed_percentage}%)."
    )
    print("- Most changes correspond to orthographic normalization "
          "such as Alif variants and Alif Maqsura.")
    print("- Canonical spellings (e.g., Ta Marbuta) were preserved "
          "to maintain compatibility with the Quranic Arabic Corpus (QAC) "
          "and Maqāyīs al-Lughah.")

DD-001 VALIDATION: ARABIC TEXT NORMALIZATION


,Metric,Value
0,Total Questions,251
1,Changed Questions,174
2,Unchanged Questions,77
3,Percentage Changed,69.32%



Representative Examples of Normalized Questions



,Question_ID,Question,Normalized_Question
1,102,من هم قوم موسى؟,من هم قوم موسي؟
2,103,من بنى الكعبة؟,من بني الكعبة؟
5,107,ما معنى الحطمة؟,ما معني الحطمة؟
6,108,من هو اخو سيدنا موسى؟,من هو اخو سيدنا موسي؟
7,109,ما معنى القارعة؟,ما معني القارعة؟
8,110,ما معنى الجاثية؟,ما معني الجاثية؟
10,112,من هي ملكة سبأ؟,من هي ملكة سبا؟
16,121,ما هو ميراث الام عن ولدها إن لم يكن له ولد؟,ما هو ميراث الام عن ولدها ان لم يكن له ولد؟
18,123,كم مدة عدّة المطلقة؟,كم مدة عدة المطلقة؟
20,127,ما هي الشجرة التي يأكل منها الكفار في النار؟,ما هي الشجرة التي ياكل منها الكفار في النار؟



Interpretation:
- Arabic text normalization modified 174 out of 251 questions (69.32%).
- Most changes correspond to orthographic normalization such as Alif variants and Alif Maqsura.
- Canonical spellings (e.g., Ta Marbuta) were preserved to maintain compatibility with the Quranic Arabic Corpus (QAC) and Maqāyīs al-Lughah.


In [10]:
# ============================================================
# Apply DD-002: Punctuation Removal
# ============================================================

# Create a copy
punctuation_df = normalized_df.copy()

# Preserve a surface-form version of the original question for
# term-level provenance. The retrieval text itself continues to use
# the normalized question exactly as in the original methodology.
punctuation_df["Original_Clean_Question"] = (
    punctuation_df["Question"]
    .apply(anlp.remove_punctuation)
)

# Apply punctuation removal to the normalized retrieval text
punctuation_df["Clean_Question"] = (
    punctuation_df["Normalized_Question"]
    .apply(anlp.remove_punctuation)
)

print("=" * 70)
print("DD-002 COMPLETED: PUNCTUATION REMOVAL")
print("=" * 70)

display(
    punctuation_df[
        [
            "Question_ID",
            "Original_Clean_Question",
            "Normalized_Question",
            "Clean_Question"
        ]
    ].head(10)
)


DD-002 COMPLETED: PUNCTUATION REMOVAL


,Question_ID,Original_Clean_Question,Normalized_Question,Clean_Question
0,101,من هم قوم شعيب,من هم قوم شعيب؟,من هم قوم شعيب
1,102,من هم قوم موسى,من هم قوم موسي؟,من هم قوم موسي
2,103,من بنى الكعبة,من بني الكعبة؟,من بني الكعبة
3,105,من هو النبي المعروف بالصبر,من هو النبي المعروف بالصبر؟,من هو النبي المعروف بالصبر
4,106,من كفل السيدة مريم,من كفل السيدة مريم؟,من كفل السيدة مريم
5,107,ما معنى الحطمة,ما معني الحطمة؟,ما معني الحطمة
6,108,من هو اخو سيدنا موسى,من هو اخو سيدنا موسي؟,من هو اخو سيدنا موسي
7,109,ما معنى القارعة,ما معني القارعة؟,ما معني القارعة
8,110,ما معنى الجاثية,ما معني الجاثية؟,ما معني الجاثية
9,111,من هم الاسباط,من هم الاسباط؟,من هم الاسباط


## Validation of DD-002

After removing punctuation and symbols, the preprocessing results are validated to verify that only punctuation marks have been eliminated while the lexical content of each question remains unchanged.

The validation reports the number of modified questions together with representative examples illustrating the effect of punctuation removal.

In [11]:
# ============================================================
# Validate DD-002: Punctuation Removal
# ============================================================

# Identify modified questions
changed_questions = punctuation_df[
    punctuation_df["Normalized_Question"] != punctuation_df["Clean_Question"]
].copy()

# Calculate statistics
total_questions = len(punctuation_df)
changed_count = len(changed_questions)
unchanged_count = total_questions - changed_count
changed_percentage = round((changed_count / total_questions) * 100, 2)

# Summary table
summary_df = pd.DataFrame({
    "Metric": [
        "Total Questions",
        "Modified Questions",
        "Unchanged Questions",
        "Percentage Modified"
    ],
    "Value": [
        total_questions,
        changed_count,
        unchanged_count,
        f"{changed_percentage}%"
    ]
})

print("=" * 70)
print("DD-002 VALIDATION: PUNCTUATION REMOVAL")
print("=" * 70)

display(summary_df)

print("\nRepresentative Examples\n")

display(
    changed_questions[
        [
            "Question_ID",
            "Normalized_Question",
            "Clean_Question"
        ]
    ].head(10)
)

print("\nInterpretation:")

print("- Arabic punctuation was successfully removed.")
print("- The lexical content of the questions was preserved.")
print("- The resulting text is ready for tokenization.")

DD-002 VALIDATION: PUNCTUATION REMOVAL


,Metric,Value
0,Total Questions,251
1,Modified Questions,251
2,Unchanged Questions,0
3,Percentage Modified,100.0%



Representative Examples



,Question_ID,Normalized_Question,Clean_Question
0,101,من هم قوم شعيب؟,من هم قوم شعيب
1,102,من هم قوم موسي؟,من هم قوم موسي
2,103,من بني الكعبة؟,من بني الكعبة
3,105,من هو النبي المعروف بالصبر؟,من هو النبي المعروف بالصبر
4,106,من كفل السيدة مريم؟,من كفل السيدة مريم
5,107,ما معني الحطمة؟,ما معني الحطمة
6,108,من هو اخو سيدنا موسي؟,من هو اخو سيدنا موسي
7,109,ما معني القارعة؟,ما معني القارعة
8,110,ما معني الجاثية؟,ما معني الجاثية
9,111,من هم الاسباط؟,من هم الاسباط



Interpretation:
- Arabic punctuation was successfully removed.
- The lexical content of the questions was preserved.
- The resulting text is ready for tokenization.


## DD-003 – Arabic Tokenization

Following text normalization and punctuation removal, each question is tokenized into a sequence of individual words.

Tokenization is a fundamental preprocessing step in Natural Language Processing (NLP), as it converts raw text into lexical units that can be analyzed independently.

Given that the QuranQA questions are short, well-formed, and whitespace-separated, this study adopts a simple whitespace-based tokenization strategy. This approach preserves the original lexical forms while providing an appropriate representation for subsequent stopword filtering, automatic query term selection, and root extraction.

In [12]:
# ============================================================
# Apply DD-003: Arabic Tokenization
# ============================================================

# Create a copy to preserve the previous stage
tokenized_df = punctuation_df.copy()

# Surface tokens are retained only for provenance
tokenized_df["Original_Tokens"] = (
    tokenized_df["Original_Clean_Question"]
    .apply(anlp.tokenize)
)

# Retrieval tokens remain the normalized tokens
tokenized_df["Tokens"] = (
    tokenized_df["Clean_Question"]
    .apply(anlp.tokenize)
)

# Normalization used here is character-level and should preserve token
# boundaries. Assert this so a future preprocessing change cannot silently
# corrupt original-term -> normalized-term provenance.
token_alignment_ok = (
    tokenized_df["Original_Tokens"].str.len()
    == tokenized_df["Tokens"].str.len()
).all()

if not token_alignment_ok:
    bad_rows = tokenized_df[
        tokenized_df["Original_Tokens"].str.len()
        != tokenized_df["Tokens"].str.len()
    ][
        [
            "Question_ID",
            "Question",
            "Original_Tokens",
            "Tokens"
        ]
    ]

    raise ValueError(
        "Original/normalized token alignment failed. "
        f"Affected questions: {len(bad_rows)}"
    )

print("=" * 70)
print("DD-003 COMPLETED: ARABIC TOKENIZATION")
print("=" * 70)

print(f"Total Questions Processed : {len(tokenized_df)}")
print(f"Token Alignment Valid     : {token_alignment_ok}")

# Display representative examples
display(
    tokenized_df[
        [
            "Question_ID",
            "Original_Tokens",
            "Tokens"
        ]
    ].head(10)
)


DD-003 COMPLETED: ARABIC TOKENIZATION
Total Questions Processed : 251
Token Alignment Valid     : True


,Question_ID,Original_Tokens,Tokens
0,101,"[من, هم, قوم, شعيب]","[من, هم, قوم, شعيب]"
1,102,"[من, هم, قوم, موسى]","[من, هم, قوم, موسي]"
2,103,"[من, بنى, الكعبة]","[من, بني, الكعبة]"
3,105,"[من, هو, النبي, المعروف, بالصبر]","[من, هو, النبي, المعروف, بالصبر]"
4,106,"[من, كفل, السيدة, مريم]","[من, كفل, السيدة, مريم]"
5,107,"[ما, معنى, الحطمة]","[ما, معني, الحطمة]"
6,108,"[من, هو, اخو, سيدنا, موسى]","[من, هو, اخو, سيدنا, موسي]"
7,109,"[ما, معنى, القارعة]","[ما, معني, القارعة]"
8,110,"[ما, معنى, الجاثية]","[ما, معني, الجاثية]"
9,111,"[من, هم, الاسباط]","[من, هم, الاسباط]"


## DD-004 – Stopword Identification

The previous notebook showed that many QuranQA questions begin with interrogative particles and contain common Arabic function words such as prepositions, pronouns, conjunctions, and auxiliary words.

Although these words are essential for constructing natural language questions, they generally do not contribute to identifying the semantic concepts required for Qur'anic passage retrieval.

Therefore, this study adopts a task-specific stopword list that combines common Arabic function words with frequently occurring interrogative particles observed in the QuranQA dataset. The resulting stopword list is used to identify non-content words before automatic query term selection.

In [13]:
# ============================================================
# Arabic Stopword List
# ============================================================

# The source list is preserved in readable Arabic orthography.
RAW_STOPWORDS = {
    "أن",
    "أو",
    "أين",
    "إلى",
    "إن",
    "التي",
    "الذي",
    "الذين",
    "اللاتي",
    "ب",
    "بماذا",
    "بين",
    "تلك",
    "ثم",
    "ذلك",
    "على",
    "عن",
    "في",
    "ك",
    "كان",
    "كانت",
    "كم",
    "كيف",
    "ل",
    "لا",
    "لم",
    "لماذا",
    "لن",
    "ما",
    "ماذا",
    "متى",
    "مع",
    "من",
    "هذا",
    "هذه",
    "هل",
    "هم",
    "هن",
    "هو",
    "هي",
    "و"
}

# Query tokens have already passed through normalize_arabic().
# Normalize the stopword set with the same function before comparison.
# This corrects cases such as إن -> ان, أين -> اين, إلى -> الي,
# على -> علي, أو -> او, and متى -> متي.
STOPWORDS = {
    anlp.normalize_arabic(word)
    for word in RAW_STOPWORDS
}

normalization_changes = sorted({
    (word, anlp.normalize_arabic(word))
    for word in RAW_STOPWORDS
    if word != anlp.normalize_arabic(word)
})

print("=" * 70)
print("ARABIC STOPWORDS LOADED")
print("=" * 70)
print(f"Source Stopwords     : {len(RAW_STOPWORDS)}")
print(f"Normalized Stopwords : {len(STOPWORDS)}")

print("\nStopwords changed by DD-001 normalization")
display(
    pd.DataFrame(
        normalization_changes,
        columns=["Source_Form", "Normalized_Form"]
    )
)


ARABIC STOPWORDS LOADED
Source Stopwords     : 41
Normalized Stopwords : 40

Stopwords changed by DD-001 normalization


,Source_Form,Normalized_Form
0,أن,ان
1,أو,او
2,أين,اين
3,إلى,الي
4,إن,ان
5,على,علي
6,متى,متي


## Applying DD-004 – Stopword Filtering

After defining the task-specific stopword list, each tokenized question is processed to identify and remove non-content words.

The remaining words represent the candidate content terms that will be considered for automatic query term selection in the subsequent stage. These candidate terms retain the primary semantic concepts expressed in the user's question while excluding common function words that contribute little to lexical retrieval.

In [14]:
# ============================================================
# Apply DD-004: Stopword Filtering
# ============================================================

# Create a copy to preserve the previous stage
filtered_df = tokenized_df.copy()

# Apply stopword filtering to the normalized token stream
filtered_df["Content_Tokens"] = (
    filtered_df["Tokens"]
    .apply(lambda tokens: anlp.remove_stopwords(tokens, STOPWORDS))
)

print("=" * 70)
print("DD-004 COMPLETED: STOPWORD FILTERING")
print("=" * 70)

print(f"Total Questions Processed : {len(filtered_df)}")

# Display representative examples
display(
    filtered_df[
        [
            "Question_ID",
            "Tokens",
            "Content_Tokens"
        ]
    ].head(10)
)


DD-004 COMPLETED: STOPWORD FILTERING
Total Questions Processed : 251


,Question_ID,Tokens,Content_Tokens
0,101,"[من, هم, قوم, شعيب]","[قوم, شعيب]"
1,102,"[من, هم, قوم, موسي]","[قوم, موسي]"
2,103,"[من, بني, الكعبة]","[بني, الكعبة]"
3,105,"[من, هو, النبي, المعروف, بالصبر]","[النبي, المعروف, بالصبر]"
4,106,"[من, كفل, السيدة, مريم]","[كفل, السيدة, مريم]"
5,107,"[ما, معني, الحطمة]","[معني, الحطمة]"
6,108,"[من, هو, اخو, سيدنا, موسي]","[اخو, سيدنا, موسي]"
7,109,"[ما, معني, القارعة]","[معني, القارعة]"
8,110,"[ما, معني, الجاثية]","[معني, الجاثية]"
9,111,"[من, هم, الاسباط]",[الاسباط]


## DD-005 – Automatic Query Term Selection

After removing common function words, the remaining tokens represent candidate content words. However, duplicate tokens or empty elements may still occur.

This stage performs a lightweight automatic query term selection by:

- Removing duplicate tokens while preserving their original order.
- Ignoring empty tokens.
- Producing a final list of candidate query terms for the subsequent root extraction stage.

More advanced linguistic filtering is intentionally deferred to the Quranic Arabic Corpus (QAC), which provides morphology-aware analysis.

In [15]:
# ============================================================
# DD-005: Automatic Query Term Selection
# ============================================================

def select_query_terms(tokens):
    """
    Select candidate query terms by removing duplicates
    while preserving the original token order.
    """

    selected = []
    seen = set()

    for token in tokens:
        token = token.strip()

        if not token:
            continue

        if token not in seen:
            selected.append(token)
            seen.add(token)

    return selected


def build_query_term_provenance(original_tokens, normalized_tokens, query_terms):
    """
    Preserve the first surface form corresponding to every selected
    normalized query term.

    Returns
    -------
    list of dict
        Ordered provenance records with token position, original term,
        and normalized term.
    """

    if len(original_tokens) != len(normalized_tokens):
        raise ValueError("Original and normalized token lists are misaligned.")

    selected_set = set(query_terms)
    seen = set()
    provenance = []

    for position, (original, normalized) in enumerate(
        zip(original_tokens, normalized_tokens)
    ):
        if normalized in selected_set and normalized not in seen:
            provenance.append({
                "Token_Position": int(position),
                "Original_Term": original,
                "Normalized_Term": normalized
            })
            seen.add(normalized)

    missing = [term for term in query_terms if term not in seen]

    if missing:
        raise ValueError(
            "Could not recover original forms for selected query terms: "
            + ", ".join(missing)
        )

    return provenance


# Create a copy
query_df = filtered_df.copy()

# Apply query term selection
query_df["Query_Terms"] = (
    query_df["Content_Tokens"]
    .apply(select_query_terms)
)

# Preserve original -> normalized query-term lineage
query_df["Query_Term_Provenance"] = query_df.apply(
    lambda row: build_query_term_provenance(
        row["Original_Tokens"],
        row["Tokens"],
        row["Query_Terms"]
    ),
    axis=1
)

print("=" * 70)
print("DD-005 COMPLETED: AUTOMATIC QUERY TERM SELECTION")
print("=" * 70)

print(f"Total Questions Processed : {len(query_df)}")

display(
    query_df[
        [
            "Question_ID",
            "Content_Tokens",
            "Query_Terms",
            "Query_Term_Provenance"
        ]
    ].head(10)
)


DD-005 COMPLETED: AUTOMATIC QUERY TERM SELECTION
Total Questions Processed : 251


,Question_ID,Content_Tokens,Query_Terms,Query_Term_Provenance
0,101,"[قوم, شعيب]","[قوم, شعيب]","[{'Token_Position': 2, 'Original_Term': 'قوم', 'Normalized_Term': 'قوم'}, {'Token_Position': 3, 'Original_Term': 'شعيب', 'Normalized_Term': 'شعيب'}]"
1,102,"[قوم, موسي]","[قوم, موسي]","[{'Token_Position': 2, 'Original_Term': 'قوم', 'Normalized_Term': 'قوم'}, {'Token_Position': 3, 'Original_Term': 'موسى', 'Normalized_Term': 'موسي'}]"
2,103,"[بني, الكعبة]","[بني, الكعبة]","[{'Token_Position': 1, 'Original_Term': 'بنى', 'Normalized_Term': 'بني'}, {'Token_Position': 2, 'Original_Term': 'الكعبة', 'Normalized_Term': 'الكعبة'}]"
3,105,"[النبي, المعروف, بالصبر]","[النبي, المعروف, بالصبر]","[{'Token_Position': 2, 'Original_Term': 'النبي', 'Normalized_Term': 'النبي'}, {'Token_Position': 3, 'Original_Term': 'المعروف', 'Normalized_Term': 'المعروف'}, {'Token_Position': 4, 'Original_Term': 'بالصبر', 'Normalized_Term': 'بالصبر'}]"
4,106,"[كفل, السيدة, مريم]","[كفل, السيدة, مريم]","[{'Token_Position': 1, 'Original_Term': 'كفل', 'Normalized_Term': 'كفل'}, {'Token_Position': 2, 'Original_Term': 'السيدة', 'Normalized_Term': 'السيدة'}, {'Token_Position': 3, 'Original_Term': 'مريم', 'Normalized_Term': 'مريم'}]"
5,107,"[معني, الحطمة]","[معني, الحطمة]","[{'Token_Position': 1, 'Original_Term': 'معنى', 'Normalized_Term': 'معني'}, {'Token_Position': 2, 'Original_Term': 'الحطمة', 'Normalized_Term': 'الحطمة'}]"
6,108,"[اخو, سيدنا, موسي]","[اخو, سيدنا, موسي]","[{'Token_Position': 2, 'Original_Term': 'اخو', 'Normalized_Term': 'اخو'}, {'Token_Position': 3, 'Original_Term': 'سيدنا', 'Normalized_Term': 'سيدنا'}, {'Token_Position': 4, 'Original_Term': 'موسى', 'Normalized_Term': 'موسي'}]"
7,109,"[معني, القارعة]","[معني, القارعة]","[{'Token_Position': 1, 'Original_Term': 'معنى', 'Normalized_Term': 'معني'}, {'Token_Position': 2, 'Original_Term': 'القارعة', 'Normalized_Term': 'القارعة'}]"
8,110,"[معني, الجاثية]","[معني, الجاثية]","[{'Token_Position': 1, 'Original_Term': 'معنى', 'Normalized_Term': 'معني'}, {'Token_Position': 2, 'Original_Term': 'الجاثية', 'Normalized_Term': 'الجاثية'}]"
9,111,[الاسباط],[الاسباط],"[{'Token_Position': 2, 'Original_Term': 'الاسباط', 'Normalized_Term': 'الاسباط'}]"


In [16]:
# ============================================================
# Save the Preprocessed Query Dataset
# ============================================================

import json

output_file = OUTPUT_DIR / "QuranQA_Preprocessed_Queries_Corrected.csv"

export_df = query_df.copy()

# Preserve the existing list columns for compatibility, while exporting the
# new provenance record as standards-compliant JSON.
export_df["Query_Term_Provenance"] = (
    export_df["Query_Term_Provenance"]
    .apply(
        lambda value: json.dumps(
            value,
            ensure_ascii=False,
            allow_nan=False
        )
    )
)

export_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("PREPROCESSED QUERY DATASET SAVED")
print("=" * 70)

print(f"Output File:\n{output_file}")
print(f"\nTotal Questions Saved: {len(export_df)}")

print(
    "\nPreservation rule: the historical "
    "'QuranQA_Preprocessed_Queries.csv' file is not overwritten."
)


PREPROCESSED QUERY DATASET SAVED
Output File:
${THESIS_WORKSPACE}\02_Methodology_Evidence\QAC and Root Extraction\Corrected_Outputs\Point_2_Query_Root_Provenance\QuranQA_Preprocessed_Queries_Corrected.csv

Total Questions Saved: 251

Preservation rule: the historical 'QuranQA_Preprocessed_Queries.csv' file is not overwritten.


# Summary and Design Implications

This notebook implemented the complete preprocessing pipeline for QuranQA user questions.

### Design Decisions

- **DD-001:** Arabic text normalization.
- **DD-002:** Punctuation and symbol removal.
- **DD-003:** Arabic tokenization.
- **DD-004:** Task-specific stopword filtering.
- **DD-005:** Automatic query term selection.

The resulting query terms preserve the primary semantic concepts expressed in each question while excluding function words that contribute little to lexical retrieval.

For reproducibility, the stopword inventory is normalized with the same DD-001 function applied to the query text before stopword comparison. The corrected export also preserves the term-level relationship between each selected surface form and its normalized query term.

The generated query term lists will serve as the input to **Notebook 5**, where each term will be analyzed using the Quranic Arabic Corpus (QAC) to identify its morphological features and corresponding Qur'anic root.
